# 02 · LightGBM Exact View

**Competition:** Kaggle Playground Series S6E9 — Predicting Electric Vehicle Purchases  
**Metric:** ROC-AUC

This notebook develops the project's high-resolution LightGBM feature view.

The central idea is to preserve fine-grained numeric structure—especially in annual income—while adding fold-safe statistical features.

The model combines:

1. high-resolution numeric features and income digit decomposition;
2. quantized income and commute features;
3. label-free frequency statistics;
4. target statistics from the external original EV dataset after exact-overlap removal;
5. a fold-fitted 600-quantile income bin;
6. three cross-fitted Target Encodings (`auto`, `10`, `100`);
7. 5-fold OOF validation;
8. three LightGBM seeds averaged at prediction level.

This view became one of the strongest individual components and later served as a core input to the ensemble.

## 1. Configuration

The notebook expects four files:

- `train.csv`
- `test.csv`
- `sample_submission.csv`
- `EV_Adoption_and_Range_Anxiety_Dataset.csv`

The fourth file is the public original EV dataset used by this feature view. Exact full-feature overlaps with **either competition train or competition test** are removed before any original-data target statistics are created.

In [ ]:
from pathlib import Path
import hashlib

import numpy as np
import pandas as pd
import lightgbm as lgb

from scipy.stats import rankdata
from sklearn.metrics import roc_auc_score
from sklearn.model_selection import StratifiedKFold
from sklearn.preprocessing import TargetEncoder

# -----------------------------
# Paths: edit if necessary
# -----------------------------
DATA_DIR = Path(".")
ORIGINAL_PATH = Path("EV_Adoption_and_Range_Anxiety_Dataset.csv")
OUTPUT_DIR = Path("outputs/02_lgb_exact")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

# -----------------------------
# Reproducible experiment setup
# -----------------------------
TARGET = "Will_Buy_EV"
ID_COL = "id"
INCOME = "Annual_Income_USD"
COMMUTE = "Daily_Commute_km"

OUTER_FOLDS = 5
INNER_FOLDS = 5
FOLD_SEED = 42
MODEL_SEEDS = [42, 152, 2026]

CATS = [
    "Gender",
    "City_Type",
    "Current_Car_Type",
    "Home_Charging_Possible",
    "Subsidy_Available",
    "Range_Anxiety_Level",
]

NUMS = [
    "Age",
    INCOME,
    COMMUTE,
    "Number_of_Cars_Owned",
    "Charging_Stations_Near_Home",
    "Charging_Stations_Near_Work",
    "Environmental_Concern_Level",
]

RAW = NUMS + CATS

## 2. Load competition data and original data

The target is mapped from `No/Yes` to `0/1`.

The external original dataset is used only after removing exact feature overlaps with all competition rows. The overlap check uses features only; no competition validation/test label is involved.

In [ ]:
def map_target(series):
    mapped = (
        series.astype("string")
        .str.strip()
        .str.lower()
        .map({"yes": 1, "no": 0, "1": 1, "0": 0, "1.0": 1, "0.0": 0})
    )
    if mapped.isna().any():
        raise ValueError("Target contains values outside Yes/No or 0/1.")
    return mapped.astype("int8")


def canonical_features(frame):
    out = frame[RAW].copy()

    for col in CATS:
        out[col] = out[col].astype("string").fillna("__MISSING__")

    for col in NUMS:
        out[col] = pd.to_numeric(out[col], errors="raise").astype("float64")

    return out


train = pd.read_csv(DATA_DIR / "train.csv")
test = pd.read_csv(DATA_DIR / "test.csv")
sample_submission = pd.read_csv(DATA_DIR / "sample_submission.csv")
original = pd.read_csv(ORIGINAL_PATH)

y = map_target(train[TARGET]).to_numpy()
original_y = map_target(original[TARGET]).to_numpy()

# Remove original rows that exactly match any competition train/test feature row.
competition_features = canonical_features(
    pd.concat([train[RAW], test[RAW]], ignore_index=True)
)
original_features = canonical_features(original)

occupied_hash = pd.util.hash_pandas_object(
    competition_features,
    index=False,
)
original_hash = pd.util.hash_pandas_object(
    original_features,
    index=False,
)

overlap_mask = original_hash.isin(occupied_hash)
overlap_removed = int(overlap_mask.sum())

original = original.loc[~overlap_mask].reset_index(drop=True)
original_y = map_target(original[TARGET]).to_numpy()

print("Train rows:", len(train))
print("Test rows:", len(test))
print("Original rows after overlap removal:", len(original))
print("Original overlaps removed:", overlap_removed)
print("Train positive rate:", y.mean())

## 3. Static label-free feature construction

The `lgb_exact` view uses a high-resolution representation.

### 3.1 Base numeric variables

For this view, `Number_of_Cars_Owned` is intentionally excluded from the base numeric matrix and from the encoding keys.

### 3.2 Digit decomposition

For every remaining numeric variable, decimal digits are extracted at powers from $10^{-4}$ through $10^3$ whenever the resulting feature is non-constant.

Income additionally receives the $10^4$ and $10^5$ integer digits.

This representation increases resolution without forcing the tree to discover every decimal boundary from the raw continuous variable.

### 3.3 Quantized income and commute

Additional keys:

- `income_floor_1`
- `income_floor_100`
- `income_floor_1000`
- `commute_integer`

### 3.4 Boundary / structural indicators

The exact view also keeps several observed structural features:

- `homecharge`
- `is_30k_spike`
- `is_income_cliff`
- `is_low_income_interval`
- `is_env_one`

These structural indicators are part of the selected feature specification.

In [ ]:
def remove_constant_and_duplicate_columns(frame):
    kept = []
    seen = {}

    for name in frame.columns:
        values = frame[name].to_numpy()

        if frame[name].nunique(dropna=False) <= 1:
            continue

        digest = hashlib.blake2b(
            np.ascontiguousarray(values).tobytes(),
            digest_size=12,
        ).digest()

        duplicate = any(
            np.array_equal(
                values,
                frame[old].to_numpy(),
                equal_nan=True,
            )
            for old in seen.get(digest, [])
        )

        if duplicate:
            continue

        seen.setdefault(digest, []).append(name)
        kept.append(name)

    return frame[kept]


all_comp = canonical_features(
    pd.concat([train[RAW], test[RAW]], ignore_index=True)
)
original_comp = canonical_features(original)

n_train = len(train)
n_total = len(all_comp)
test_idx = np.arange(n_train, n_total)

numeric_features = {}
raw_keys = {}
original_keys = {}

# Base variables and raw encoding keys.
for col in RAW:
    if col == "Number_of_Cars_Owned":
        continue

    values = all_comp[col].to_numpy()
    original_values = original_comp[col].to_numpy()

    if col in NUMS:
        numeric_features[col] = values.astype("float32")

    raw_keys[col] = values
    original_keys[col] = original_values

income = all_comp[INCOME].to_numpy()
commute = all_comp[COMMUTE].to_numpy()

# High-resolution digit decomposition.
for col in [c for c in NUMS if c != "Number_of_Cars_Owned"]:
    values = all_comp[col].to_numpy()

    for power in range(-4, 4):
        name = f"{col}_digit_{power}"
        digit = (values // (10.0 ** power) % 10).astype("int8")

        if np.unique(digit).size > 1:
            numeric_features[name] = digit.astype("float32")
            raw_keys[name] = digit

# Extra high-order income digits.
for power in [4, 5]:
    name = f"income_digit_{power}"
    digit = (income.astype("int64") // (10 ** power) % 10).astype("float32")

    numeric_features[name] = digit
    raw_keys[name] = digit

# Exact-view income ladder.
for divisor in [1, 100, 1000]:
    name = f"income_floor_{divisor}"

    floor_all = np.floor(income / divisor).astype("float32")
    floor_original = np.floor(
        original_comp[INCOME].to_numpy() / divisor
    )

    numeric_features[name] = floor_all
    raw_keys[name] = floor_all
    original_keys[name] = floor_original

# Integer commute.
commute_integer = np.floor(commute).astype("float32")
numeric_features["commute_integer"] = commute_integer
raw_keys["commute_integer"] = commute_integer
original_keys["commute_integer"] = np.floor(
    original_comp[COMMUTE].to_numpy()
)

# Structural numeric indicators.
env = all_comp["Environmental_Concern_Level"].to_numpy()
home_possible = (
    all_comp["Home_Charging_Possible"] == "Yes"
).to_numpy(dtype="int8")

numeric_features["homecharge"] = (
    all_comp["Charging_Stations_Near_Home"].to_numpy()
    + 5 * home_possible
).astype("float32")

numeric_features["is_30k_spike"] = (
    income == 30000
).astype("float32")

numeric_features["is_income_cliff"] = (
    income >= 170537
).astype("float32")

numeric_features["is_low_income_interval"] = (
    (income >= 38000) & (income <= 42000)
).astype("float32")

numeric_features["is_env_one"] = (
    env == 1
).astype("float32")

numeric_static = remove_constant_and_duplicate_columns(
    pd.DataFrame(numeric_features)
)

print("Static numeric features:", numeric_static.shape[1]) 

## 4. Encoding keys, global frequencies, and original-data means

The raw key values are factorized into integer category codes. The factorization itself is label-free.

Two additional static statistics are added.

### 4.1 Competition frequency

For each key $k$,

$$
\mathrm{freq}(v)
=
\frac{\#\{i : k_i=v\}}
{N_{\mathrm{train}}+N_{\mathrm{test}}}.
$$

This frequency is computed over competition train + test and therefore uses **no target labels**.

### 4.2 Original-data mean target rate

For eligible keys that also exist in the original dataset,

$$
\mathrm{original\_mean}(v)
=
\frac{\sum_{i \in \mathrm{original}}
y_i\,\mathbf{1}(k_i=v)}
{\sum_{i \in \mathrm{original}}
\mathbf{1}(k_i=v)}.
$$

Unseen values fall back to the original-data prior.

This is an external supervised feature source. It is not derived from competition validation labels.

In [ ]:
key_codes = {}
original_mean_features = {}

original_prior = original_y.mean()

for name, values in raw_keys.items():
    series = pd.Series(values)

    codes, uniques = pd.factorize(
        series,
        sort=False,
    )

    if len(uniques) <= 1:
        continue

    key_codes[name] = codes.astype("int32")

    # Original-data target means are available only for keys
    # with an explicitly constructed original counterpart.
    if name in original_keys:
        stats_frame = pd.DataFrame({
            "key": original_keys[name],
            "y": original_y,
        }).dropna(subset=["key"])

        stats = (
            stats_frame
            .groupby("key", observed=True)["y"]
            .agg(["sum", "count"])
        )

        original_mean_features[f"original_mean_{name}"] = (
            series.map(stats["sum"] / stats["count"])
            .fillna(original_prior)
            .to_numpy(dtype="float32")
        )

keys = remove_constant_and_duplicate_columns(
    pd.DataFrame(key_codes)
)

# Keep original-mean features only for surviving keys.
original_mean_features = {
    name: values
    for name, values in original_mean_features.items()
    if name.removeprefix("original_mean_") in keys.columns
}

# Label-free train+test frequencies.
frequency_features = {}

for col in keys.columns:
    freq_map = keys[col].value_counts(normalize=True)

    frequency_features[f"freq_{col}"] = (
        keys[col]
        .map(freq_map)
        .to_numpy(dtype="float32")
    )

static_features = remove_constant_and_duplicate_columns(
    pd.concat(
        [
            numeric_static,
            pd.DataFrame(frequency_features),
            pd.DataFrame(original_mean_features),
        ],
        axis=1,
    )
).astype("float32")

print("Encoding keys:", keys.shape[1])
print("Final static features before fold-specific transforms:", static_features.shape[1])

## 5. Fold-safe Target Encoding

The outer validation fold must never contribute its target values to its own features.

For each outer fold:

1. only outer-training labels are available to the encoder;
2. training rows receive **inner cross-fitted** target encodings;
3. outer validation rows are transformed from the encoder fitted on outer training;
4. test rows are transformed from the same fitted encoder.

Three smoothing settings are used:

- `smooth="auto"`
- `smooth=10`
- `smooth=100`

For a category value $v$, a smoothed target encoding can be viewed schematically as

$$
\mathrm{TE}(v)
=
\frac{n_v\bar y_v+\lambda\mu}
{n_v+\lambda},
$$

where $\mu$ is a global prior and $\lambda$ controls shrinkage.

The exact implementation is `sklearn.preprocessing.TargetEncoder`.

In [ ]:
def crossfit_target_encoding(
    key_frame,
    y,
    train_idx,
    valid_idx,
    test_idx,
    smooth_values,
    inner_folds,
    seed,
):
    train_parts = []
    valid_parts = []
    test_parts = []
    feature_names = []

    for smooth in smooth_values:
        encoder = TargetEncoder(
            target_type="binary",
            smooth=smooth,
            cv=inner_folds,
            shuffle=True,
            random_state=seed,
        )

        train_encoded = encoder.fit_transform(
            key_frame.iloc[train_idx],
            y[train_idx],
        ).astype("float32")

        valid_encoded = encoder.transform(
            key_frame.iloc[valid_idx]
        ).astype("float32")

        test_encoded = encoder.transform(
            key_frame.iloc[test_idx]
        ).astype("float32")

        train_parts.append(train_encoded)
        valid_parts.append(valid_encoded)
        test_parts.append(test_encoded)

        feature_names.extend([
            f"TE_{col}_{smooth}"
            for col in key_frame.columns
        ])

    return (
        np.column_stack(train_parts),
        np.column_stack(valid_parts),
        np.column_stack(test_parts),
        feature_names,
    )

## 6. Build one outer-fold feature matrix

One additional feature is created **inside each outer fold**: a 600-quantile income bin.

The quantile edges are fitted only on outer-training income values, then applied to outer validation and test. The resulting bin is used both as a numeric feature and as an additional Target-Encoding key.

This keeps the discretization rule fold-local.

In [ ]:
def build_fold_matrices(
    train_idx,
    valid_idx,
    seed,
):
    # Start from static label-free features.
    X_train = static_features.iloc[train_idx].copy()
    X_valid = static_features.iloc[valid_idx].copy()
    X_test = static_features.iloc[test_idx].copy()

    # Fold-fitted 600-quantile income bin.
    income_all = all_comp[INCOME].to_numpy(dtype="float64")

    edges = np.unique(
        np.quantile(
            income_all[train_idx],
            np.linspace(0, 1, 601),
        )
    )

    income_bin = np.searchsorted(
        edges[1:-1],
        income_all,
        side="right",
    ).astype("int32")

    X_train["income_quantile_600"] = income_bin[train_idx]
    X_valid["income_quantile_600"] = income_bin[valid_idx]
    X_test["income_quantile_600"] = income_bin[test_idx]

    fold_keys = keys.copy()
    fold_keys["income_quantile_600"] = income_bin

    (
        te_train,
        te_valid,
        te_test,
        te_names,
    ) = crossfit_target_encoding(
        key_frame=fold_keys,
        y=y,
        train_idx=train_idx,
        valid_idx=valid_idx,
        test_idx=test_idx,
        smooth_values=["auto", 10.0, 100.0],
        inner_folds=INNER_FOLDS,
        seed=seed,
    )

    X_train[te_names] = te_train
    X_valid[te_names] = te_valid
    X_test[te_names] = te_test

    # Remove constants according to outer-training rows only.
    keep = [
        col
        for col in X_train.columns
        if X_train[col].nunique(dropna=False) > 1
    ]

    X_train = X_train[keep]
    X_valid = X_valid[keep]
    X_test = X_test[keep]

    return X_train, X_valid, X_test

## 7. LightGBM model

The selected `lgb_exact` LightGBM configuration is:

- learning rate: `0.02`
- max depth: `5`
- leaves: `32`
- minimum data per leaf: `10`
- feature fraction: `0.30293`
- bagging fraction: `0.812763`
- L1: `0.07094`
- L2: `2.03303`
- max bin: `1024`
- early stopping: `500`

Three seeds are trained. Each seed uses the same outer folds but changes both the Target-Encoding cross-fit randomness and the LightGBM seed.

In [ ]:
def fit_lgb_exact(
    X_train,
    y_train,
    X_valid,
    y_valid,
    X_test,
    seed,
):
    params = {
        "objective": "binary",
        "metric": "auc",
        "learning_rate": 0.02,
        "max_depth": 5,
        "num_leaves": 32,
        "min_data_in_leaf": 10,
        "feature_fraction": 0.30293,
        "bagging_freq": 0,
        "lambda_l1": 0.07094,
        "lambda_l2": 2.03303,
        "max_bin": 1024,
        "feature_pre_filter": False,
        "verbosity": -1,
        "seed": seed,
        "deterministic": True,
        "force_col_wise": True,
    }

    dtrain = lgb.Dataset(
        X_train,
        label=y_train,
    )

    dvalid = lgb.Dataset(
        X_valid,
        label=y_valid,
        reference=dtrain,
    )

    model = lgb.train(
        params,
        dtrain,
        num_boost_round=20000,
        valid_sets=[dvalid],
        callbacks=[
            lgb.early_stopping(
                500,
                verbose=False,
            ),
            lgb.log_evaluation(500),
        ],
    )

    best_iteration = model.best_iteration or 20000

    valid_pred = model.predict(
        X_valid,
        num_iteration=best_iteration,
    )

    test_pred = model.predict(
        X_test,
        num_iteration=best_iteration,
    )

    importance = pd.DataFrame({
        "feature": X_train.columns,
        "gain": model.feature_importance(
            importance_type="gain"
        ),
    })

    return (
        valid_pred,
        test_pred,
        best_iteration,
        importance,
    )

## 8. Five-fold OOF training with three seeds

The outer split is fixed once:

$$
\text{StratifiedKFold}(5,\ \text{shuffle=True},\ \text{random\_state}=42).
$$

For each seed and outer fold:

1. construct fold-safe features;
2. fit LightGBM with early stopping on the outer validation fold;
3. save validation predictions into the OOF vector;
4. predict the competition test set.

Seed predictions are averaged at prediction level.

In [ ]:
outer_cv = StratifiedKFold(
    n_splits=OUTER_FOLDS,
    shuffle=True,
    random_state=FOLD_SEED,
)

folds = list(
    outer_cv.split(train, y)
)

fold_id = np.empty(
    len(train),
    dtype="int8",
)

for fold, (_, valid_idx) in enumerate(
    folds,
    start=1,
):
    fold_id[valid_idx] = fold


seed_oof_predictions = []
seed_test_fold_predictions = []
training_rows = []
importance_frames = []

for seed in MODEL_SEEDS:
    seed_oof = np.zeros(
        len(train),
        dtype="float64",
    )

    seed_test_folds = []

    for fold, (train_idx, valid_idx) in enumerate(
        folds,
        start=1,
    ):
        X_train, X_valid, X_test = build_fold_matrices(
            train_idx=train_idx,
            valid_idx=valid_idx,
            seed=seed,
        )

        (
            valid_pred,
            test_pred,
            best_iteration,
            importance,
        ) = fit_lgb_exact(
            X_train=X_train,
            y_train=y[train_idx],
            X_valid=X_valid,
            y_valid=y[valid_idx],
            X_test=X_test,
            seed=seed,
        )

        seed_oof[valid_idx] = valid_pred
        seed_test_folds.append(test_pred)

        fold_auc = roc_auc_score(
            y[valid_idx],
            valid_pred,
        )

        training_rows.append({
            "seed": seed,
            "fold": fold,
            "auc": fold_auc,
            "best_iteration": best_iteration,
            "n_features": X_train.shape[1],
        })

        importance["seed"] = seed
        importance["fold"] = fold
        importance_frames.append(importance)

        print(
            f"seed={seed} "
            f"fold={fold} "
            f"AUC={fold_auc:.6f} "
            f"best={best_iteration} "
            f"features={X_train.shape[1]}"
        )

    seed_oof_predictions.append(
        seed_oof
    )

    seed_test_fold_predictions.append(
        np.stack(seed_test_folds)
    )

## 9. Aggregate seeds, evaluate OOF, and build fold-normalized ranks

Raw predictions are averaged across the three seeds.

For later rank ensembles, the project also created **fold-normalized OOF ranks**. Each validation fold is ranked independently:

$$
R_i
=
\frac{\operatorname{rank}(p_i)-0.5}
{n_{\text{fold}}}.
$$

Test predictions are ranked separately for each outer fold and then averaged. This mirrors the representation later used by the ensemble pipeline.

In [ ]:
def normalized_rank(values):
    values = np.asarray(values)

    return (
        rankdata(
            values,
            method="average",
        )
        - 0.5
    ) / len(values)


# Average raw OOF predictions across seeds.
oof_pred = np.mean(
    seed_oof_predictions,
    axis=0,
)

# Shape: seeds x folds x test_rows
test_seed_fold = np.stack(
    seed_test_fold_predictions
)

# Average seeds first, retaining one test vector per outer fold.
test_fold_pred = np.mean(
    test_seed_fold,
    axis=0,
)

# Raw test prediction: average outer folds.
test_pred = np.mean(
    test_fold_pred,
    axis=0,
)

# Fold-normalized OOF ranks.
oof_rank = np.zeros(
    len(train),
    dtype="float64",
)

test_rank = np.zeros(
    len(test),
    dtype="float64",
)

for fold, (_, valid_idx) in enumerate(
    folds
):
    oof_rank[valid_idx] = normalized_rank(
        oof_pred[valid_idx]
    )

    test_rank += (
        normalized_rank(
            test_fold_pred[fold]
        )
        / OUTER_FOLDS
    )


raw_oof_auc = roc_auc_score(
    y,
    oof_pred,
)

rank_oof_auc = roc_auc_score(
    y,
    oof_rank,
)

fold_metrics = pd.DataFrame(
    training_rows
)

print(f"Raw OOF AUC:  {raw_oof_auc:.6f}")
print(f"Rank OOF AUC: {rank_oof_auc:.6f}")
print()
print("Reproduction target:")
print("  raw OOF  ≈ 0.946146")
print("  rank OOF ≈ 0.946150")

## 10. Fold stability and seed stability

The final component is the average of three closely performing seed models. The objective of the seed ensemble is variance reduction rather than hyperparameter selection.

In [ ]:
seed_summary = []

for seed, seed_oof in zip(
    MODEL_SEEDS,
    seed_oof_predictions,
):
    fold_scores = []

    for _, valid_idx in folds:
        fold_scores.append(
            roc_auc_score(
                y[valid_idx],
                seed_oof[valid_idx],
            )
        )

    seed_summary.append({
        "seed": seed,
        "mean_fold_auc": np.mean(fold_scores),
        "fold_auc_std": np.std(fold_scores),
        "pooled_oof_auc": roc_auc_score(
            y,
            seed_oof,
        ),
    })

seed_summary = pd.DataFrame(
    seed_summary
)

seed_summary

## 11. Feature importance

Gain importance is aggregated over every trained seed/fold model. It is used only as a descriptive diagnostic; no feature is selected from validation performance in this notebook.

In [ ]:
importance_all = pd.concat(
    importance_frames,
    ignore_index=True,
)

importance_summary = (
    importance_all
    .groupby("feature", as_index=False)["gain"]
    .mean()
    .sort_values(
        "gain",
        ascending=False,
    )
)

importance_summary.head(30)

## 12. Save reproducible outputs

The saved OOF/test files are designed to be consumed by the later ensemble notebooks.

In [ ]:
oof_output = pd.DataFrame({
    ID_COL: train[ID_COL],
    "y_true": y,
    "fold": fold_id,
    "prediction": oof_pred,
    "prediction_rank": oof_rank,
})

test_output = pd.DataFrame({
    ID_COL: test[ID_COL],
    "prediction": test_pred,
    "prediction_rank": test_rank,
})

submission = sample_submission.copy()
submission[TARGET] = test_pred

oof_output.to_csv(
    OUTPUT_DIR / "oof_lgb_exact.csv",
    index=False,
)

test_output.to_csv(
    OUTPUT_DIR / "test_lgb_exact.csv",
    index=False,
)

fold_metrics.to_csv(
    OUTPUT_DIR / "fold_metrics.csv",
    index=False,
)

seed_summary.to_csv(
    OUTPUT_DIR / "seed_metrics.csv",
    index=False,
)

importance_summary.to_csv(
    OUTPUT_DIR / "feature_importance.csv",
    index=False,
)

submission.to_csv(
    OUTPUT_DIR / "submission_lgb_exact.csv",
    index=False,
)

print("Saved outputs to:", OUTPUT_DIR)

## 13. Interpretation

The exact view improved substantially over the Logistic baseline because it changed both the model family and the representation:

- LightGBM learns nonlinear thresholds and interactions;
- digit decomposition exposes fine numeric structure;
- quantization makes important income boundaries easier to express;
- cross-fitted Target Encoding injects supervised category information without outer-fold leakage;
- train+test frequencies add label-free prevalence information;
- original-data target means provide an independent supervised statistic after overlap removal;
- averaging three seeds reduces prediction variance.

The important methodological point is that label-derived competition features are built inside the validation structure. The model is not evaluated on target statistics computed from its own validation labels.

This component later served as one of the strongest inputs to the project's rank ensembles.

## Result

- Raw OOF AUC: **0.946146**
- Rank-normalized OOF AUC: **0.946150**

This component is used in the final ensemble analysis in
`06_final_research.ipynb`.